# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

frame = con.sql("""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS early_impressions,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS early_clicks,
  SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') * 1.0
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15'), 0) AS early_ctr,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS early_avg_position,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS early_active_days,
  SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15') AS late_impressions
FROM march
WHERE gsc_data_available IS TRUE
  AND hash(content_hash_id) % 50 = 0
GROUP BY 1, 2
""").df()

df = frame[frame["early_impressions"] >= 20].copy()
df["late_impressions"] = df["late_impressions"].fillna(0)
df["label_declining"] = ((df["late_impressions"] / 16) < 0.8 * (df["early_impressions"] / 15)).astype(int)
features = ["early_impressions", "early_clicks", "early_ctr", "early_avg_position", "early_active_days"]
print(df.shape, "declining rate:", round(df["label_declining"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2191, 9) declining rate: 0.329


In [3]:
# repeatable results: DuckDB can return rows in a different order on each run
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
X = df[features].fillna(0)
y = df["label_declining"]
groups = df["client_hash_id"]

from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, score, k):
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return np.asarray(y_true)[idx].mean()

def peer_baseline(train_idx, apply_idx):
    """ML-07 rule: peer CTR per position bucket learned on train_idx, scored on apply_idx."""
    cut = lambda s: pd.cut(s, [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)
    tr = df.iloc[train_idx].dropna(subset=["early_avg_position", "early_ctr"])
    tr = tr[tr["early_avg_position"] <= 20]
    sums = tr.groupby(cut(tr["early_avg_position"]))[["early_clicks", "early_impressions"]].sum()
    peer = sums["early_clicks"] / sums["early_impressions"]
    ap = df.iloc[apply_idx]
    score = (cut(ap["early_avg_position"]).map(peer) * ap["early_impressions"] - ap["early_clicks"]).clip(lower=0)
    return score.fillna(0).values

print(df.shape, "declining rate:", round(y.mean(), 3))

(2191, 9) declining rate: 0.329


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Finding 1: "The concentration signal is independently associated with decline risk" (Abstract; raw gap 45.2% vs 33.8% in Section 4.3).**
Methodology question: where does the label come from, and does the timing line up? The label compares April with March impressions, while the query-portfolio table covers a fixed 90-day window. I would ask which dates that window covers relative to 1 April, because if it reaches into April the concentration feature could partly see the label period. The paper says it excludes the *_last30 columns, which is a good control; I would ask the same question of the remaining portfolio columns. I would also ask which analysis shows the signal is "independent" of the others, for example a model with and without it, beyond the raw gap and the importance chart.

**Finding 2: "Top-50 precision of 0.62 exceeds the baseline's 0.52" (Abstract; Section 4.5).**
Methodology question: does the validation design support the claim? With 50 pages, a 10-point gap is close to normal sampling noise (about 7 points), so I would ask whether the gap holds across repeated splits or comes with an interval. The split is described as both time-aware and page-grouped, so I would ask how pages in the test month are kept apart from training pages and whether the baseline was scored on exactly the same pages.

What I am copying from the paper: every number carries a run label, the language is observed / directional / decision-support, and it makes no causal claims.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
def models():
    return {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Random forest": RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=0, n_jobs=-1),
    }
BASE = "Baseline rule (peer CTR from train folds only)"

def run_split(splits, label):
    oof = {k: np.zeros(len(df)) for k in list(models()) + [BASE]}
    for tr, te in splits:
        for k, m in models().items():
            m.fit(X.iloc[tr], y.iloc[tr])
            oof[k][te] = m.predict_proba(X.iloc[te])[:, 1]
        oof[BASE][te] = peer_baseline(tr, te)
    rows = [{"split": label, "model": k, "AUC": roc_auc_score(y, p),
             "Precision@50": precision_at_k(y, p, 50), "Precision@100": precision_at_k(y, p, 100)}
            for k, p in oof.items()]
    return pd.DataFrame(rows), oof

before_splits = list(KFold(n_splits=5, shuffle=True, random_state=0).split(X))   # random: ignores clients
after_splits  = list(GroupKFold(n_splits=5).split(X, y, groups))                   # grouped by client

for k, (tr, te) in enumerate(before_splits):
    print(f"random fold {k}: test clients that also appear in training = {len(set(groups.iloc[te]) & set(groups.iloc[tr]))} of {groups.iloc[te].nunique()}")
for k, (tr, te) in enumerate(after_splits):
    print(f"grouped fold {k}: clients in both = {len(set(groups.iloc[te]) & set(groups.iloc[tr]))}")

t_before, oof_before = run_split(before_splits, "BEFORE: random split")
t_after,  oof_after  = run_split(after_splits,  "AFTER: grouped by client")
print("base rate:", round(y.mean(), 3))
pd.concat([t_before, t_after]).round(3)

random fold 0: test clients that also appear in training = 24 of 24
random fold 1: test clients that also appear in training = 23 of 24
random fold 2: test clients that also appear in training = 24 of 27
random fold 3: test clients that also appear in training = 23 of 24
random fold 4: test clients that also appear in training = 23 of 25
grouped fold 0: clients in both = 0
grouped fold 1: clients in both = 0
grouped fold 2: clients in both = 0
grouped fold 3: clients in both = 0
grouped fold 4: clients in both = 0
base rate: 0.329


,split,model,AUC,Precision@50,Precision@100
0,BEFORE: random split,Logistic regression,0.598,0.44,0.52
1,BEFORE: random split,Random forest,0.632,0.54,0.50
2,BEFORE: random split,Baseline rule (peer CTR from train folds only),0.570,0.38,0.42
0,AFTER: grouped by client,Logistic regression,0.560,0.38,0.39
1,AFTER: grouped by client,Random forest,0.555,0.22,0.32
2,AFTER: grouped by client,Baseline rule (peer CTR from train folds only),0.572,0.40,0.43


Before (random split) vs after (grouped by client): in the random split, 23 to 24 of the test clients in every fold also appeared in training, so the learned models saw each client before. Their numbers fell when I grouped by client: random forest AUC 0.632 → 0.555 and Precision@50 0.54 → 0.22; logistic regression AUC 0.598 → 0.560 and Precision@50 0.44 → 0.38. The baseline rule hardly moved (AUC 0.570 → 0.572, Precision@50 0.38 → 0.40), because it only learns three peer-CTR values. So part of what the random split showed was client-specific patterns that do not carry over to new clients, and I keep the grouped numbers as my honest ones. Under the grouped split no model clearly beats the baseline; with 50 pages, Precision@50 moves about 7 points by chance. This is an observed result on one month and about 2,200 pages.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

| Feature | How it is built | Window | Knowable at the decision moment (end of 15 March)? |
|---|---|---|---|
| early_impressions | sum of daily impressions | 1–15 March | Yes |
| early_clicks | sum of daily clicks | 1–15 March | Yes |
| early_ctr | early clicks / early impressions | 1–15 March | Yes |
| early_avg_position | mean of daily average position | 1–15 March | Yes |
| early_active_days | days with impressions > 0 | 1–15 March | Yes |

Never used as inputs: late_impressions and label_declining (16–31 March), trend_direction, trend_pct. One built-in link I noticed: the label compares the late rate with 0.8 × the early rate, so early_impressions appears in both the label rule and the features. That is not future information, but it means high-early-volume pages are more likely to fall back (regression to the mean).

In [8]:
def grouped_auc(cols, target):
    Xc = df[cols].fillna(0)
    p = np.zeros(len(df))
    for tr, te in after_splits:
        m = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=0, n_jobs=-1)
        m.fit(Xc.iloc[tr], target.iloc[tr])
        p[te] = m.predict_proba(Xc.iloc[te])[:, 1]
    return roc_auc_score(target, p)

print("honest AUC:", round(grouped_auc(features, y), 3))

df["late_vs_early"] = (df["late_impressions"] / 16) / (df["early_impressions"] / 15)   # built from the label window, on purpose
print("AUC with a deliberately leaky column:", round(grouped_auc(features + ["late_vs_early"], y), 3))
df = df.drop(columns=["late_vs_early"])

y_shuffled = pd.Series(np.random.default_rng(0).permutation(y.values), index=y.index)
print("AUC with shuffled labels (should be near 0.5):", round(grouped_auc(features, y_shuffled), 3))

honest AUC: 0.555
AUC with a deliberately leaky column: 1.0
AUC with shuffled labels (should be near 0.5): 0.487


In [9]:
all_idx = np.arange(len(df))
s_in = peer_baseline(all_idx, all_idx)    # ML-07 style: peer CTR from all pages
print("baseline, peer CTR from ALL pages (ML-07 way): AUC", round(roc_auc_score(y, s_in), 3), " P@50", round(precision_at_k(y, s_in, 50), 3))
print(t_after[t_after["model"] == BASE][["AUC", "Precision@50"]].round(3).to_string(index=False), " <- peer CTR from train folds only")

baseline, peer CTR from ALL pages (ML-07 way): AUC 0.57  P@50 0.38
  AUC  Precision@50
0.572           0.4  <- peer CTR from train folds only


In [10]:
p = pd.Series(oof_after["Logistic regression"], index=df.index)
show = ["early_impressions", "early_clicks", "early_ctr", "early_avg_position", "early_active_days"]

top50 = df.loc[p.sort_values(ascending=False).head(50).index]
fp = top50[top50["label_declining"] == 0].copy()
fp["rate_change_pct"] = (((fp["late_impressions"] / 16) / (fp["early_impressions"] / 15) - 1) * 100).round(1)   # for review only, not an input
print("False positives (high score, did not decline):", len(fp), "of top 50")
display(fp[show + ["rate_change_pct"]].head(8))

bottom50 = df.loc[p.sort_values().head(50).index]
fn = bottom50[bottom50["label_declining"] == 1].copy()
fn["rate_change_pct"] = (((fn["late_impressions"] / 16) / (fn["early_impressions"] / 15) - 1) * 100).round(1)
print("False negatives (lowest scores that did decline):", len(fn), "of bottom 50")
display(fn[show + ["rate_change_pct"]].head(5))

False positives (high score, did not decline): 31 of top 50


,early_impressions,early_clicks,early_ctr,early_avg_position,early_active_days,rate_change_pct
1537,30687.0,33.0,0.001075,6.834084,15,-16.9
255,18554.0,0.0,0.000000,3.391745,14,-1.7
2068,11086.0,4.0,0.000361,3.273758,15,40.9
2086,12499.0,4.0,0.000320,27.661718,15,11.6
594,17283.0,5.0,0.000289,32.721259,15,28.3
303,14572.0,9.0,0.000618,25.248303,15,12.9
1914,8920.0,4.0,0.000448,6.436707,13,76.9
1711,6761.0,3.0,0.000444,6.195396,15,93.9


False negatives (lowest scores that did decline): 7 of bottom 50


,early_impressions,early_clicks,early_ctr,early_avg_position,early_active_days,rate_change_pct
1713,27.0,2.0,0.074074,9.160000,5,-27.1
525,6210.0,50.0,0.008052,25.934197,15,-53.7
601,2211.0,42.0,0.018996,20.125702,15,-44.3
1272,6392.0,62.0,0.009700,13.670696,15,-65.4
435,1669.0,27.0,0.016177,20.422489,15,-37.9


Leakage results (measured on the grouped split): the honest AUC is 0.555. Adding a column built from the label window (late_vs_early) pushes AUC to 1.0, which is the fingerprint of leakage, and it shows my five early-window features did not already contain that kind of shortcut. With shuffled labels the AUC is 0.487, close to 0.5, so the pipeline does not produce signal out of nothing. Taking the baseline's peer CTR from all pages gave AUC 0.570 and Precision@50 0.38, against 0.572 and 0.40 when it came from the training folds only, so that choice did not leak in any visible way.

Failure examples: 31 of the logistic regression's top 50 did not decline. Many are large pages (6,761 to 30,687 early impressions) with a CTR close to zero, and several grew in the second half (+12% to +94%), so low CTR on a big page is not the same as a declining page. One false positive had a rate change of −16.9%, just inside the −20% label cut-off. The false negatives are mostly pages with a moderate CTR for their position that still fell 38% to 65%, plus one very small page (27 impressions), so the five features miss declines that do not start with a poor CTR. Limit: all checks use one month and about 2,200 pages, so these are observed and directional.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

**My boldest sentence (before):** "My rule picks pages that are declining, with Precision@50 of 0.38, better than the 0.329 base rate."
**Rewritten:** "Observed: of the 50 pages my rule ranked first, 38% declined in 16–31 March, against 32.9% for all pages. With 50 pages that gap is within chance, so it is a decision-support starting point, not evidence that the rule works."

| My earlier claim | What the evidence supports | Rewritten claim |
|---|---|---|
| ML-07: position matters for CTR | CTR was higher at positions 1–3 but not steady across buckets (MIXED) | Directional: higher CTR at the top positions, not a steady pattern |
| ML-08: no model clearly beat the baseline | Grouped split: logistic regression AUC 0.560 / Precision@50 0.38, random forest 0.555 / 0.22, baseline 0.572 / 0.40; the random split flattered the learned models (forest AUC 0.632) | Observed in this sample: under a grouped split no model clearly beat the rule; a different month or more pages could change it |
| ML-08: CTR and position matter most to the forest | Importance came from one fold and moved between runs (0.107 → 0.095) | Directional: CTR and position ranked highest in my runs; the values vary |
| The label means "declining" | A rule on 15 vs 16 days of impressions; includes regression to the mean | Decision-support proxy for pages worth reviewing, not proof of decline |

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.